In [1]:
import pandas as pd
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

In [2]:
df = pd.read_csv("music_reccomendation_dataset.csv")

Clean column names (strip BOM / whitespace)

In [3]:
df.columns = df.columns.str.strip().str.lstrip("\ufeff")

In [4]:
print("Shape:", df.shape)
print(df.head(3))

Shape: (2420, 5)
          Song-Name                       Singer/Artists  \
0       Aankh Marey  Kumar Sanu, Mika Singh, Neha Kakkar   
1         Coca Cola             Neha Kakkar, Tony Kakkar   
2  Apna Time Aayega                        Ranveer Singh   

                    Genre  Album/Movie User-Rating  
0          BollywoodDance       Simmba      8.8/10  
1  BollywoodDanceRomantic  Luka Chuppi      9.0/10  
2          BollywoodDance    Gully Boy      9.7/10  


Convert rating 9.2/10 to float 9.2

In [5]:
df["Rating"] = df["User-Rating"].str.replace("/10", "", regex=False).astype(float)

In [7]:
df["soup"] = ( df["Genre"].fillna("") + " " + df["Genre"].fillna("") + " " +
              df["Singer/Artists"].fillna("") +
              " " + df["Album/Movie"].fillna(""))

Normalise text

In [8]:
df["soup"] = df["soup"].str.lower().str.replace(r"[^a-z0-9 ]", " ", regex=True)

In [ ]:
df = df.reset_index(drop=True)

BUILD TF-IDF MATRIX & COSINE SIMILARITY

In [9]:
tfidf = TfidfVectorizer(stop_words="english")
tfidf_matrix = tfidf.fit_transform(df["soup"])

In [10]:
cosine_sim = cosine_similarity(tfidf_matrix, tfidf_matrix)

In [11]:
song_index = pd.Series(df.index, index=df["Song-Name"].str.lower()).drop_duplicates()

In [15]:
def recommend(song_name: str, top_n: int = 10):
    key = song_name.strip().lower()
    if key not in song_index:
        matches = [s for s in song_index.index if key in s]
        if not matches:
            print(f'"{song_name}" not found in dataset.')
            print("Try one of these songs:")
            print(df["Song-Name"].sample(10).tolist())
            return pd.DataFrame()
        key = matches[0]
        print(f'Showing results for closest match: "{df.loc[song_index[key], "Song-Name"]}"')

    idx = song_index[key]

    sim_scores = list(enumerate(cosine_sim[idx]))
    sim_scores = sorted(sim_scores, key=lambda x: x[1], reverse=True)
    sim_scores = sim_scores[1 : top_n + 1]
    song_indices   = [i[0] for i in sim_scores]
    sim_values     = [round(i[1], 4) for i in sim_scores]

    result = df.iloc[song_indices][["Song-Name", "Singer/Artists", "Genre", "Album/Movie", "Rating"]].copy()
    result["Similarity"] = sim_values
    result = result.sort_values(by="Similarity", ascending=False).reset_index(drop=True)
    result.index += 1
    return result

In [17]:
if __name__ == "__main__":
    query = "Coca Cola" #change this song according to your wish

    print(f"\nTop 10 recommendations for: '{query}'\n")
    recs = recommend(query, top_n=10)
    display(recs)


Top 10 recommendations for: 'Coca Cola'



,Song-Name,Singer/Artists,Genre,Album/Movie,Rating,Similarity
1,Mile Ho Tum (Reprise),"Neha Kakkar, Tony Kakkar",BollywoodRomantic,Fever,9.0,0.5281
2,Mohabbat Nasha Hai,"Neha Kakkar, Tony Kakkar",BollywoodSad,Hate Story 4,9.4,0.4976
3,Helicopter,"Neha Kakkar, Tony Kakkar",BollywoodDance,Ranchi Diaries,9.3,0.4771
4,La La La (Baazaar),"Bilal Saeed, Neha Kakkar",BollywoodDanceRomantic,Baazaar,9.5,0.4740
5,Sawan Aaya Hai (Unplugged Version),"Neha Kakkar, Tony Kakkar",BollywoodSad,Creature 3D,9.3,0.4466
6,Khambe Jaisi Khadi Hai,Udit Narayan,BollywoodDanceRomantic,Dil,9.0,0.4094
7,Poster Lagwa Do,"Mika Singh, Sunanda Sharma",BollywoodDance,Luka Chuppi,8.4,0.3971
8,Dhoom Machale,Sunidhi Chauhan,BollywoodDanceRomantic,Dhoom,9.0,0.3771
9,Ek Dilruba Hai,Udit Narayan,BollywoodDanceRomantic,Bewafaa,8.9,0.3721
10,Dil Ashkon Mein,"Sonu Kakkar, Tony Kakkar",BollywoodRomance,Fever,9.4,0.3689
